# Data Cleaning: Ticket Prices and Surcharges

This notebook performs the cleaning process for the `3_ticket_prices_and_surcharges.csv` dataset in-memory.

In [1]:
import pandas as pd
import numpy as np

In [2]:

# Load the dataset
df = pd.read_csv('../data/processed/3_ticket_prices_and_surcharges.csv')
print("Original Shape:", df.shape)
df.head()

Original Shape: (14355, 25)


,month,conflict_phase,airline,iata_code,country,region,airline_type,route_class,avg_route_km,base_fare_usd,...,load_factor_pct,fuel_cost_pct_opex,yoy_price_change_pct,km_range,surcharge_band,fuel_surcharge_usd_surcharge_policy,brent_crude_usd_barrel,jet_fuel_usd_barrel_surcharge_policy,surcharge_as_pct_base,yoy_surcharge_change_pct
0,2019-01,Pre-Pandemic Baseline,ANA,NH,Japan,Asia,Flag Carrier,Long-Haul,8500,1179.91,...,79.4,0.209,NaN,4501–9000 km,NaN,NaN,NaN,NaN,NaN,NaN
1,2019-02,Pre-Pandemic Baseline,ANA,NH,Japan,Asia,Flag Carrier,Long-Haul,8500,1176.08,...,89.0,0.216,NaN,4501–9000 km,NaN,NaN,NaN,NaN,NaN,NaN
2,2019-03,Pre-Pandemic Baseline,ANA,NH,Japan,Asia,Flag Carrier,Long-Haul,8500,1133.88,...,64.1,0.246,NaN,4501–9000 km,NaN,NaN,NaN,NaN,NaN,NaN
3,2019-04,Pre-Pandemic Baseline,ANA,NH,Japan,Asia,Flag Carrier,Long-Haul,8500,1237.95,...,75.1,0.225,NaN,4501–9000 km,NaN,NaN,NaN,NaN,NaN,NaN
4,2019-05,Pre-Pandemic Baseline,ANA,NH,Japan,Asia,Flag Carrier,Long-Haul,8500,1270.08,...,85.8,0.268,NaN,4501–9000 km,NaN,NaN,NaN,NaN,NaN,NaN


### 1. Remove Duplicates

In [3]:
duplicates = df.duplicated().sum()
print(f"Found {duplicates} duplicate rows.")
if duplicates > 0:
    df = df.drop_duplicates()
    print("Duplicates removed. New Shape:", df.shape)

Found 0 duplicate rows.


### 2. Handle Missing Values

In [4]:
print("Missing values before cleaning:")
print(df.isnull().sum()[df.isnull().sum() > 0])

# Fill missing categorical data
if 'surcharge_band' in df.columns:
    df['surcharge_band'] = df['surcharge_band'].fillna('Unknown')

if 'route_class' in df.columns:
    df['route_class'] = df['route_class'].fillna('Unknown')

# Fill missing numeric data (e.g., yoy percentages)
num_cols_to_fill_zero = ['yoy_price_change_pct', 'yoy_surcharge_change_pct', 'surcharge_as_pct_base']
for col in num_cols_to_fill_zero:
    if col in df.columns:
        df[col] = df[col].fillna(0)

print("\nMissing values after cleaning:")
print(df.isnull().sum().sum())

Missing values before cleaning:
yoy_price_change_pct                    1980
surcharge_band                          3480
fuel_surcharge_usd_surcharge_policy     3480
brent_crude_usd_barrel                  3480
jet_fuel_usd_barrel_surcharge_policy    3480
surcharge_as_pct_base                   3480
yoy_surcharge_change_pct                4980
dtype: int64

Missing values after cleaning:
10440


### 3. Data Type Conversions & Standardization

In [5]:
# Standardize strings
string_cols = df.select_dtypes(include=['object']).columns
for col in string_cols:
    df[col] = df[col].astype(str).str.strip()

# Ensure date format
if 'month' in df.columns:
    df['month'] = pd.to_datetime(df['month'], format='%Y-%m', errors='coerce').dt.strftime('%Y-%m')

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 14355 entries, 0 to 14354
Data columns (total 25 columns):
 #   Column                                Non-Null Count  Dtype  
---  ------                                --------------  -----  
 0   month                                 14355 non-null  str    
 1   conflict_phase                        14355 non-null  str    
 2   airline                               14355 non-null  str    
 3   iata_code                             14355 non-null  str    
 4   country                               14355 non-null  str    
 5   region                                14355 non-null  str    
 6   airline_type                          14355 non-null  str    
 7   route_class                           14355 non-null  str    
 8   avg_route_km                          14355 non-null  int64  
 9   base_fare_usd                         14355 non-null  float64
 10  fuel_surcharge_usd                    14355 non-null  float64
 11  taxes_fees_usd            

/var/folders/z5/dvvhg5811tg1_7qfcnjnd_b40000gn/T/ipykernel_64467/4175898254.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  string_cols = df.select_dtypes(include=['object']).columns
